# Machine Translation & Quality Estimation — Practical Session

In this session we follow the natural life-cycle of a machine translation:

1. **Translate** — we play with two open, multilingual MT engines on a set of *deliberately difficult* Spanish sentences and watch them struggle. This is where the **importance of good translation** becomes visible.
2. **Evaluate (with a reference)** — we score outputs with the classic reference-based metrics **BLEU** and **BLEURT**, and see what they can and cannot capture.
3. **Estimate quality (without a reference)** — in the real world (live chat, news feeds, on-the-fly translation) we almost never have a human reference. That is why we turn to **Quality Estimation (QE)** with **COMET-Kiwi** and **XCOMET**.

Everything stays in the **Spanish &harr; English** pair so the three parts tell one story.

# Part 1 — Machine Translation Models

Machine Translation (MT) is the task of automatically translating text from one language to another. Neural MT (NMT) is today's dominant paradigm, and although early systems were built for a single language pair, modern NMT models are naturally **multilingual**.

Let's play with two widely used open multilingual systems, **M2M100** and **NLLB**, and translate Spanish into English.

In [ ]:
!pip install --upgrade pip
!pip install transformers sentencepiece sacremoses nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.2 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 9.1 MB/s  0:00:00


In [ ]:
source_sentences = [
    "La biblioteca abre a las nueve de la mañana.",   # 0 easy control
    "Tirar la casa por la ventana.",                  # 1 idiom
    "Estoy en el quinto pino.",                       # 2 idiom
    "No tengo pelos en la lengua.",                   # 3 idiom
    "Cuesta un ojo de la cara.",                      # 4 idiom
    "Dejó el libro en el banco.",                     # 5 ambiguity (bench / bank)
]

## M2M100

M2M100 (*Beyond English-Centric Multilingual Machine Translation*, Fan et al.) is a multilingual encoder-decoder model. The target language is forced as the first generated token via `forced_bos_token_id`.

In [ ]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

m2m_model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M")
m2m_tok   = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")
m2m_tok.src_lang = "es"

def m2m_translate(text, tgt="en"):
    enc = m2m_tok(text, return_tensors="pt")
    gen = m2m_model.generate(**enc, forced_bos_token_id=m2m_tok.get_lang_id(tgt))
    return m2m_tok.batch_decode(gen, skip_special_tokens=True)[0]

for s in source_sentences:
    print("ES:", s)
    print("EN:", m2m_translate(s), "\n")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/3.71M [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.14k [00:00<?, ?B/s]

ES: La biblioteca abre a las nueve de la mañana.
EN: The library opens at nine in the morning. 

ES: Tirar la casa por la ventana.
EN: Take the house out of the window. 

ES: Estoy en el quinto pino.
EN: I am in the fifth pine. 

ES: No tengo pelos en la lengua.
EN: I have no hair in my tongue. 

ES: Cuesta un ojo de la cara.
EN: It costs one eye on the face. 

ES: Dejó el libro en el banco.
EN: He left the book in the bank. 



## NLLB

NLLB (*No Language Left Behind*, Costa-jussà et al.) covers 200 languages. Unlike M2M100 it uses BCP-47 codes (Spanish = `spa_Latn`, English = `eng_Latn`). The small helper below reads the target-language id in a way that works across `transformers` versions.

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

nllb_tok   = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M", src_lang="spa_Latn")
nllb_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")

def nllb_lang_id(tok, code):
    # Newer transformers dropped tokenizer.lang_code_to_id; fall back gracefully.
    if hasattr(tok, "lang_code_to_id"):
        return tok.lang_code_to_id[code]
    return tok.convert_tokens_to_ids(code)

def nllb_translate(text, tgt="eng_Latn"):
    enc = nllb_tok(text, return_tensors="pt")
    gen = nllb_model.generate(**enc,
                              forced_bos_token_id=nllb_lang_id(nllb_tok, tgt),
                              max_length=60)
    return nllb_tok.batch_decode(gen, skip_special_tokens=True)[0]

for s in source_sentences:
    print("ES:", s)
    print("EN:", nllb_translate(s), "\n")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

ES: La biblioteca abre a las nueve de la mañana.
EN: The library opens at nine in the morning. 

ES: Tirar la casa por la ventana.
EN: Throw the house out the window. 

ES: Estoy en el quinto pino.
EN: I'm on the fifth pine. 

ES: No tengo pelos en la lengua.
EN: I don't have any hair on my tongue. 

ES: Cuesta un ojo de la cara.
EN: It costs an eye to the face. 

ES: Dejó el libro en el banco.
EN: He left the book in the bank. 



### Compare the two engines side by side

In [ ]:
print(f"{'Spanish source':40s} | {'M2M100':40s} | {'NLLB':40s}")
print("-" * 126)
for s in source_sentences:
    print(f"{s:40s} | {m2m_translate(s):40s} | {nllb_translate(s):40s}")

Spanish source                           | M2M100                                   | NLLB                                    
------------------------------------------------------------------------------------------------------------------------------
La biblioteca abre a las nueve de la mañana. | The library opens at nine in the morning. | The library opens at nine in the morning.
Tirar la casa por la ventana.            | Take the house out of the window.        | Throw the house out the window.         
Estoy en el quinto pino.                 | I am in the fifth pine.                  | I'm on the fifth pine.                  
No tengo pelos en la lengua.             | I have no hair in my tongue.             | I don't have any hair on my tongue.     
Cuesta un ojo de la cara.                | It costs one eye on the face.            | It costs an eye to the face.            
Dejó el libro en el banco.               | He left the book in the bank.            | He left the book in

**Discuss.** Notice how the idioms (1–4) tend to come out *literal* or only partly idiomatic, and how the ambiguous *banco* in sentence 5 forces the engine to **guess** between *bench* and *bank* with no way to ask for clarification.

This raises the obvious question: **different engines give different outputs so which one is best?** To answer that, we need a way to *measure* translation quality.

### Your turn

Write a Spanish sentence you think will be hard (an idiom, a pun, an ambiguous word, a long subordinate clause) and translate it with both engines. Did either of them get it right?

In [ ]:
my_sentence = "..."   # <- your difficult Spanish sentence here

print("M2M100:", m2m_translate(my_sentence))
print("NLLB  :", nllb_translate(my_sentence))

M2M100: ...
NLLB  : - What ?


# Part 2 — Machine Translation Evaluation (with a reference)

There are two ways to judge an MT system. **Human evaluation** is the gold standard but is slow and expensive. **Automatic evaluation** uses metrics that compare the MT output against one or more **human reference translations**. Let's look at the two most common reference-based metrics.

## BLEU

BLEU compares the n-grams of the candidate translation against a human reference and returns a score: the more n-grams they share, the higher the score. It is fast, language-independent and extremely widely used — but it only counts **surface word overlap**, so it is blind to meaning, to synonyms, and (for unigrams) to word order.

In [ ]:
import nltk
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu

def bleu(ref, gen):
    """Pairwise BLEU using the NLTK implementation (as in the original tutorial)."""
    ref_bleu = [[r.split()] for r in ref]
    gen_bleu = [g.split() for g in gen]
    cc = SmoothingFunction()
    return corpus_bleu(ref_bleu, gen_bleu, weights=(0, 1, 0, 0), smoothing_function=cc.method4)

**A faithful but non-literal translation.** Both sentences mean the same thing, yet BLEU drops because *greatest/biggest* and *time/era* are different surface words.

In [ ]:
Reference = "Climate change is one of the greatest challenges of our time."
Candidate = "Climate change is one of the biggest challenges of our era."
print("BLEU:", bleu([Reference], [Candidate]))

BLEU: 0.7


**An idiom translated literally.** This is what an engine produced for *Cuesta un ojo de la cara.* It is wrong, and BLEU correctly gives it a very low score **but only because we happened to have the human reference**.

In [ ]:
Reference = "It costs an arm and a leg."
Candidate = "It costs an eye of the face."
print("BLEU:", bleu([Reference], [Candidate]))

BLEU: 0.3333333333333333


**Word order.** BLEU (at the unigram level) gives these two the same score even though the second one means something completely different — a classic illustration of its blindness to order.

In [ ]:
Reference = "The guard arrived late because of the rain."
Candidate = "The rain arrived late because of the guard."
print("BLEU:", bleu([Reference], [Candidate]))

BLEU: 0.5714285714285714


## BLEURT

To get past pure surface overlap, researchers built **trained** metrics such as **BLEURT**. BLEURT is a regression model (based on BERT/RemBERT) fine-tuned on human ratings; it takes a reference and a candidate and returns a score reflecting how fluent the candidate is and how well it conveys the meaning of the reference. Scores are roughly in `[0, 1]`.

In [ ]:
!pip install --upgrade pip
!git clone https://github.com/google-research/bleurt.git
%cd bleurt
!pip install .
!wget https://storage.googleapis.com/bleurt-oss-21/BLEURT-20.zip
!unzip BLEURT-20.zip
%cd ..

Cloning into 'bleurt'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 134 (delta 0), reused 17 (delta 0), pack-reused 116 (from 1)
Receiving objects: 100% (134/134), 31.28 MiB | 12.52 MiB/s, done.
Resolving deltas: 100% (49/49), done.
/content/bleurt
Processing ./.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for BLEURT: filename=bleurt-0.0.2-py3-none-any.whl size=16456845 sha256=52650475a6ed93170d1082804327d83a9d5ac7390248cabadb688b27fa354c21
  Stored in directory: /tmp/pip-ephem-wheel-cache-xrut9bx4/wheels/27/29/e7/dd83f91837d3166c90e8f10b032d942ee54bc533fc36c98ecb
Successfully built BLEURT
--2026-06-11 14:42:58--  https://storage.googleapis.com/bleurt-oss-21/BLEURT-20.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 142.250.101.207, 142.250.141.207, 142.251.2.2

BLEURT rewards the *meaning-preserving* paraphrase that BLEU penalised, and still punishes the nonsense candidate.

In [ ]:
from bleurt import score

scorer = score.BleurtScorer("bleurt/BLEURT-20")

references = ["Climate change is one of the greatest challenges of our time."]

good = ["Climate change is one of the biggest challenges of our era."]   # paraphrase
bad  = ["Climate change is a recipe for chocolate cake."]                # nonsense

print("BLEURT (paraphrase):", scorer.score(references=references, candidates=good))
print("BLEURT (nonsense)  :", scorer.score(references=references, candidates=bad))

BLEURT (paraphrase): [0.8709949851036072]
BLEURT (nonsense)  : [0.3106611371040344]


## The catch: every metric so far needs a reference

BLEU and BLEURT and even modern reference-based COMET all require a **human reference translation** to compare against. That is fine in a research benchmark, but think about where MT is actually deployed:

* a live customer-support chat being translated as it is typed,
* a news feed translated into 50 languages the moment it is published,
* a translator's CAT tool flagging segments that need attention.

In none of these cases does a human reference exist — if it did, we would not need the MT in the first place. **This is the gap that Quality Estimation fills.**

# Part 3 — Quality Estimation (no reference)

The goal of **quality estimation (QE)** is to evaluate the quality of a translation **without having access to a reference**. Because QE only looks at the *source* and the *machine translation*, it can run in real time and be used to:

* select the best output when several engines are available;
* warn the end user about the reliability of the translation;
* decide whether a segment can be published as-is, needs post-editing, or must be re-translated.

QE can be done at **document**, **sentence** and **word** level. We will use two state-of-the-art models from the [COMET](https://github.com/Unbabel/COMET) framework:

| Granularity | Model | What it gives you |
|---|---|---|
| Sentence level | `Unbabel/wmt22-cometkiwi-da` (**COMET-Kiwi**) | one quality score in `[0, 1]` per segment |
| Word / span level | `Unbabel/XCOMET-XL` (**XCOMET**) | a segment score **and** the individual error spans, each tagged *minor / major / critical* |

To make the errors easy to see, most of the Spanish &rarr; English translations below are **deliberately wrong**. Watch how the scores drop and how XCOMET points to exactly where the errors are.

## Setup

COMET is the `unbabel-comet` package (we need `>=2.2.0` for XCOMET error spans).

> **Runtime note.** These models are large (COMET-Kiwi &asymp; 2.3 GB, XCOMET-XL &asymp; 3.5 B parameters / &asymp; 14 GB). Use a **GPU runtime** (Colab: *Runtime &rarr; Change runtime type &rarr; GPU*).

In [ ]:
# Run this BEFORE importing comet.
# COMET is PyTorch-only; this stops `transformers` from importing TensorFlow,
# which avoids the protobuf clash introduced by the BLEURT (TensorFlow) install in Part 2.
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

In [ ]:
!pip install --upgrade pip
!pip install "unbabel-comet>=2.2.0"

Both `Unbabel/wmt22-cometkiwi-da` and `Unbabel/XCOMET-XL` are **gated models** (CC-BY-NC-SA, for non-commercial evaluation). Before downloading them you must:

1. While logged in to Hugging Face, **accept the licence** on each model page:
   * https://huggingface.co/Unbabel/wmt22-cometkiwi-da
   * https://huggingface.co/Unbabel/XCOMET-XL
2. **Log in** below with an access token (https://huggingface.co/settings/tokens).

In [ ]:
from huggingface_hub import notebook_login
notebook_login()          # paste a token with at least 'read' permission

# Non-interactive alternative:
# from huggingface_hub import login
# login(token="hf_xxx")

### Our Spanish &rarr; English QE examples

The **source** is Spanish, the **machine translation** is English. One translation is correct (control); the rest contain typical MT error types. For QE we pass the model **only** `src` and `mt` — never a reference.

| # | Error type | What is wrong |
|---|---|---|
| A | *none (control)* | faithful translation &rarr; should score **high** |
| B | negation flip | the MT inserts *not*, inverting the meaning |
| C | fluent but unfaithful | *agua* (water) is rendered as *wine* — fluent, completely wrong |
| D | multiple lexical errors | *gato* &rarr; dog, *calle* &rarr; river, *tranquilamente* &rarr; quickly |
| E | omission / under-translation | the time and the reason are dropped entirely |

In [ ]:
# Source is Spanish (src), machine translation is English (mt).
# COMET QE models take ONLY src + mt — no reference is provided.
examples = [
    {   # A — correct control
        "src": "El cambio climático es uno de los mayores desafíos de nuestro tiempo.",
        "mt":  "Climate change is one of the greatest challenges of our time.",
    },
    {   # B — negation flip (meaning inverted)
        "src": "Reducir estos conflictos es importante para la conservación.",
        "mt":  "Reducing these conflicts is not important for conservation.",
    },
    {   # C — fluent but unfaithful: agua (water) -> wine
        "src": "Los médicos recomiendan beber ocho vasos de agua al día.",
        "mt":  "Doctors recommend drinking eight glasses of wine a day.",
    },
    {   # D — several lexical errors: cat->dog, street->river, calmly->quickly
        "src": "El gato negro cruzó la calle tranquilamente.",
        "mt":  "The black dog crossed the river quickly.",
    },
    {   # E — omission: time + reason dropped
        "src": "La reunión se pospuso hasta el próximo martes debido al mal tiempo.",
        "mt":  "The meeting was postponed.",
    },
]

labels = ["A correct", "B negation", "C water->wine", "D lexical", "E omission"]
for lab, ex in zip(labels, examples):
    print(f"[{lab}]")
    print("  ES:", ex["src"])
    print("  EN:", ex["mt"])

[A correct]
  ES: El cambio climático es uno de los mayores desafíos de nuestro tiempo.
  EN: Climate change is one of the greatest challenges of our time.
[B negation]
  ES: Reducir estos conflictos es importante para la conservación.
  EN: Reducing these conflicts is not important for conservation.
[C water->wine]
  ES: Los médicos recomiendan beber ocho vasos de agua al día.
  EN: Doctors recommend drinking eight glasses of wine a day.
[D lexical]
  ES: El gato negro cruzó la calle tranquilamente.
  EN: The black dog crossed the river quickly.
[E omission]
  ES: La reunión se pospuso hasta el próximo martes debido al mal tiempo.
  EN: The meeting was postponed.


## Sentence-level QE with COMET-Kiwi

`Unbabel/wmt22-cometkiwi-da` is a **reference-free** regression model (built on InfoXLM). It takes the source and the translation and returns one score in `[0, 1]`, where values close to **1** mean high quality and close to **0** mean poor quality. The COMET API is the same for every model: `download_model` &rarr; `load_from_checkpoint` &rarr; `model.predict(...)`.

In [ ]:
from comet import download_model, load_from_checkpoint
import torch

kiwi_path = download_model("Unbabel/wmt22-cometkiwi-da")
kiwi_model = load_from_checkpoint(kiwi_path)

ImportError: cannot import name 'runtime_version' from 'google.protobuf' (/usr/local/lib/python3.12/dist-packages/google/protobuf/__init__.py)

In [ ]:
gpus = 1 if torch.cuda.is_available() else 0
kiwi_output = kiwi_model.predict(examples, batch_size=8, gpus=gpus)

print("COMET-Kiwi quality scores (reference-free)\n")
for lab, score in zip(labels, kiwi_output.scores):
    print(f"  {lab:15s}  {score:.4f}")

print(f"\nSystem (average) score: {kiwi_output.system_score:.4f}")

**What to look for.** The control (A) should sit near the top. The negation flip (B) and the *water &rarr; wine* substitution (C) are *fluent* English, yet COMET-Kiwi penalises them because they are unfaithful to the source — and it does so **with no reference**, exactly the situation we are in when MT is deployed live.

### A real QE use case: ranking competing MT outputs

Recall the Part 1 question — *which engine is best?* With QE we can answer it per sentence, with no reference, by scoring every candidate and keeping the highest.

In [ ]:
ranking_src = "La biblioteca abre a las nueve de la mañana de lunes a viernes."

candidates = [
    "The library opens at nine in the morning from Monday to Friday.",   # faithful
    "The library opens at nine in the morning from Monday to Sunday.",   # Friday -> Sunday
    "The bookshop closes at nine at night on weekends.",                 # many errors
]

ranking_data = [{"src": ranking_src, "mt": c} for c in candidates]
ranking_out = kiwi_model.predict(ranking_data, batch_size=8, gpus=gpus)

ranked = sorted(zip(candidates, ranking_out.scores), key=lambda x: x[1], reverse=True)

print("Source (ES):", ranking_src, "\n")
print("Candidates ranked by COMET-Kiwi:\n")
for rank, (cand, score) in enumerate(ranked, start=1):
    print(f"  {rank}. [{score:.4f}]  {cand}")

print(f"\nBest candidate selected by QE:\n  {ranked[0][0]}")

## Word-level QE with XCOMET

A sentence score tells you *that* a translation is bad, not *where* or *why*. **XCOMET** (`Unbabel/XCOMET-XL`, 3.5 B parameters, XLM-R XL) returns a segment score **and** the **error spans**, each with an MQM severity:

* **minor** — small fluency/style issue, meaning preserved;
* **major** — clear error that affects the meaning;
* **critical** — severe error (e.g. a negation flip or a dangerous mistranslation).

Given only **source + MT**, it acts as a reference-free, word-level QE system.

In [ ]:
from comet import download_model, load_from_checkpoint

xcomet_path = download_model("Unbabel/XCOMET-XL")
xcomet_model = load_from_checkpoint(xcomet_path)

We reuse the examples richest in errors — **B** (negation), **C** (*water &rarr; wine*) and **D** (cat/street/calmly).

In [ ]:
span_examples = [examples[1], examples[2], examples[3]]   # B, C, D
span_labels   = ["B negation", "C water->wine", "D lexical"]

xcomet_out = xcomet_model.predict(span_examples, batch_size=8, gpus=gpus)

for lab, score in zip(span_labels, xcomet_out.scores):
    print(f"{lab:15s}  segment score = {score:.4f}")

The detected spans live in `model_output.metadata.error_spans`: a list with one entry per segment, each a list of spans carrying the offending `text`, a `severity` and a `confidence`.

In [ ]:
import pprint
pprint.pprint(xcomet_out.metadata.error_spans)

Let's render the spans inline, the way Unbabel's MQM viewer does, by wrapping each flagged span in an `<error severity=...>` tag inside the translation.

In [ ]:
def annotate(mt, spans):
    """Rebuild the MT string with <error severity=...> ... </error> markers.

    Each span dict from XCOMET typically contains 'start', 'end', 'severity' and
    'confidence'. We use character offsets when present, and fall back to a plain
    text match otherwise so the helper is robust across comet versions.
    """
    if not spans:
        return mt + "    (no errors detected)"

    if all("start" in s and "end" in s for s in spans):
        out = mt
        for s in sorted(spans, key=lambda x: x["start"], reverse=True):
            sev = s.get("severity", "?")
            seg = out[s["start"]:s["end"]]
            out = out[:s["start"]] + f"<error severity='{sev}'>{seg}</error>" + out[s["end"]:]
        return out

    out = mt
    for s in spans:
        sev = s.get("severity", "?")
        txt = s.get("text", "")
        if txt:
            out = out.replace(txt, f"<error severity='{sev}'>{txt}</error>", 1)
    return out


for lab, ex, score, spans in zip(
        span_labels, span_examples, xcomet_out.scores, xcomet_out.metadata.error_spans):
    print(f"=== {lab}  (segment score = {score:.4f}) ===")
    print("  ES :", ex["src"])
    print("  MT :", annotate(ex["mt"], spans))
    print()

Now the output is **explainable**: the negation flip is flagged as *critical*, *wine* is a *critical* mistranslation of *agua*, and *dog / river / quickly* are the problem words in D — all without a reference. This is the fine-grained feedback word-level QE adds on top of a sentence score, and it is exactly what a post-editor needs.

## Exercise

Write **your own** Spanish source sentences and (wrong) English machine translations — try a negation, a wrong number, a dropped clause, or a fluent-but-unfaithful substitution. Then:

1. score them with **COMET-Kiwi** and check whether the scores match how bad you think each is;
2. run **XCOMET** on the worst one and see whether the error spans land where you expected.

In [ ]:
my_examples = [
    {"src": "...",   # your Spanish source
     "mt":  "..."},  # your (wrong) English translation
    # add more pairs here
]

# Sentence-level QE
my_kiwi = kiwi_model.predict(my_examples, batch_size=8, gpus=gpus)
for ex, score in zip(my_examples, my_kiwi.scores):
    print(f"[{score:.4f}]  {ex['mt']}")

# Word-level QE (uncomment once you have at least one real pair)
# my_xcomet = xcomet_model.predict(my_examples, batch_size=8, gpus=gpus)
# for ex, spans in zip(my_examples, my_xcomet.metadata.error_spans):
#     print(annotate(ex["mt"], spans))

# Wrap-up

| | Needs a reference? | Output |
|---|---|---|
| BLEU | **yes** | n-gram overlap with a human translation |
| BLEURT | **yes** | trained meaning-aware similarity to a reference |
| COMET-Kiwi (`wmt22-cometkiwi-da`) | **no** | one quality score per segment |
| XCOMET (`XCOMET-XL`) | **no** (QE mode) | segment score **+** word-level error spans with severities |

The story of the session:

* **Translation is hard** — idioms and ambiguity in Part 1 show why quality cannot be assumed, and why different engines disagree.
* **Reference-based evaluation works but needs a human translation** — BLEU is blind to meaning, BLEURT fixes that, but both are unusable when no reference exists.
* **QE estimates quality from the source and the translation alone** — so it works live; COMET-Kiwi gives a score, and XCOMET goes further and tells you *which words* are wrong and *how badly*.